In [8]:
# Import necessary libraries
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl

plt.style.use('ggplot')
%matplotlib inline

In [9]:
path = r"C:\Users\Kuinox\Documents\parquet_output"
addresses_file = os.path.join(path, 'addresses.parquet')
symbols_file = os.path.join(path, 'symbols.parquet')
tracesamples_file = os.path.join(path, 'tracesamples.parquet')

In [ ]:
addresses_df = pl.scan_parquet(addresses_file)
symbols_df = pl.scan_parquet(symbols_file)
tracesamples_df = pl.scan_parquet(tracesamples_file)

print("\n--- Addresses Schema ---")
display(addresses_df.collect_schema())
print("\n--- Symbols Schema ---")
display(symbols_df.collect_schema())
print("\n--- Trace Samples Schema ---")
display(tracesamples_df.collect_schema())

addresses_count = addresses_df.select(pl.len()).collect().item()
symbols_count = symbols_df.select(pl.len()).collect().item()
tracesamples_count = tracesamples_df.select(pl.len()).collect().item()

print(f"Addresses count: {addresses_count:,}")
print(f"Symbols count: {symbols_count:,}")
print(f"Trace samples count: {tracesamples_count:,}")

Addresses count: 232,917,414
Symbols count: 720
Trace samples count: 212,496,770


In [11]:
BRANCH_FLAGS = {
    'BRANCH': 1 << 0,
    'CALL': 1 << 1,
    'RETURN': 1 << 2,
    'CONDITIONAL': 1 << 3,
    'SYSCALLRET': 1 << 4,
    'ASYNC': 1 << 5,
    'INTERRUPT': 1 << 6,
    'TX_ABORT': 1 << 7,
    'TRACE_BEGIN': 1 << 8,
    'TRACE_END': 1 << 9,
    'IN_TX': 1 << 10,
    'VMENTRY': 1 << 11,
    'VMEXIT': 1 << 12
}

trace_schema = tracesamples_df.collect_schema()
print("Available columns:", list(trace_schema.keys()))

# Let's examine the Flags column instead of Event
flags_values = tracesamples_df.select("Flags").limit(10).collect()
print("\nSample Flags values:")
display(flags_values)

print("\nAnalyzing branch types...")
branch_analysis = (
    tracesamples_df
    .with_columns([
        (pl.col('Flags') & BRANCH_FLAGS['BRANCH'] > 0).alias('is_branch'),
        (pl.col('Flags') & BRANCH_FLAGS['CALL'] > 0).alias('is_call'),
        (pl.col('Flags') & BRANCH_FLAGS['RETURN'] > 0).alias('is_return'),
        (pl.col('Flags') & BRANCH_FLAGS['CONDITIONAL'] > 0).alias('is_conditional'),
        (pl.col('Flags') & BRANCH_FLAGS['SYSCALLRET'] > 0).alias('is_syscallret'),
        (pl.col('Flags') & BRANCH_FLAGS['ASYNC'] > 0).alias('is_async'),
        (pl.col('Flags') & BRANCH_FLAGS['INTERRUPT'] > 0).alias('is_interrupt'),
        (pl.col('Flags') & BRANCH_FLAGS['TX_ABORT'] > 0).alias('is_tx_abort'),
        (pl.col('Flags') & BRANCH_FLAGS['TRACE_BEGIN'] > 0).alias('is_trace_begin'),
        (pl.col('Flags') & BRANCH_FLAGS['TRACE_END'] > 0).alias('is_trace_end'),
        (pl.col('Flags') & BRANCH_FLAGS['IN_TX'] > 0).alias('is_in_tx'),
        (pl.col('Flags') & BRANCH_FLAGS['VMENTRY'] > 0).alias('is_vmentry'),
        (pl.col('Flags') & BRANCH_FLAGS['VMEXIT'] > 0).alias('is_vmexit')
    ])
    .select([
        pl.sum('is_branch').alias('branch_count'),
        pl.sum('is_call').alias('call_count'),
        pl.sum('is_return').alias('return_count'),
        pl.sum('is_conditional').alias('conditional_count'),
        pl.sum('is_syscallret').alias('syscallret_count'),
        pl.sum('is_async').alias('async_count'),
        pl.sum('is_interrupt').alias('interrupt_count'),
        pl.sum('is_tx_abort').alias('tx_abort_count'),
        pl.sum('is_trace_begin').alias('trace_begin_count'),
        pl.sum('is_trace_end').alias('trace_end_count'),
        pl.sum('is_in_tx').alias('in_tx_count'),
        pl.sum('is_vmentry').alias('vmentry_count'),
        pl.sum('is_vmexit').alias('vmexit_count')
    ])
    .collect()
)

print("\nBranch type counts:")
display(branch_analysis)

branch_counts_list = []
for flag_name, _ in BRANCH_FLAGS.items():
    col_name = f"{flag_name.lower()}_count"
    if col_name in branch_analysis.columns:
        count = branch_analysis[0, col_name]
        branch_counts_list.append({"branch_type": flag_name, "count": count})

display(pl.DataFrame(branch_counts_list).sort("count", descending=True))

timeline_analysis = (
    tracesamples_df
    .with_columns([
        (pl.col('Time').cast(pl.Int64) // 1000).alias('second'),
        (pl.col('Flags') & BRANCH_FLAGS['BRANCH'] > 0).alias('is_branch'),
        (pl.col('Flags') & BRANCH_FLAGS['CALL'] > 0).alias('is_call'),
        (pl.col('Flags') & BRANCH_FLAGS['RETURN'] > 0).alias('is_return'),
        (pl.col('Flags') & BRANCH_FLAGS['CONDITIONAL'] > 0).alias('is_conditional')
    ])
    .group_by('second')
    .agg([
        pl.sum('is_branch').alias('branch_count'),
        pl.sum('is_call').alias('call_count'),
        pl.sum('is_return').alias('return_count'),
        pl.sum('is_conditional').alias('conditional_count'),
        pl.sum('InsnCnt').alias('instruction_count')
    ])
    .sort('second')
    .collect()
)

display(timeline_analysis.head(10))

timeline_analysis = timeline_analysis.with_columns([
    (pl.col('branch_count') / pl.col('instruction_count')).alias('branch_ratio'),
    (pl.col('call_count') / pl.col('instruction_count')).alias('call_ratio'),
    (pl.col('return_count') / pl.col('instruction_count')).alias('return_ratio'),
    (pl.col('conditional_count') / pl.col('instruction_count')).alias('conditional_ratio')
])

Available columns: ['Id', 'PerfId', 'Pid', 'Tid', 'Time', 'Cpu', 'Ip', 'Addr', 'Period', 'InsnCnt', 'CycCnt', 'Weight', 'Cpumode', 'AddrCorrelatesSym', 'Event', 'MachinePid', 'Vcpu']


ColumnNotFoundError: Flags

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
Parquet SCAN [C:\Users\Kuinox\Documents\parquet_output\tracesamples.parquet]
PROJECT */17 COLUMNS

In [ ]:
# Visualizations
insn_cyc_results = (
    tracesamples_df
    .with_columns([
        pl.col('Time').cast(pl.Int64).alias('timestamp_ms'),
        (pl.col('Time').cast(pl.Int64) // 1000).alias('second')
    ])
    .group_by('second')
    .agg([
        pl.sum('InsnCnt').alias('instruction_count'),
        pl.sum('CycCnt').alias('cycle_count')
    ])
    .sort('second')
    .collect()
    .with_columns([(pl.col('instruction_count') / pl.col('cycle_count')).alias('IPC')])
)

seconds = insn_cyc_results['second'].to_list()
instructions = insn_cyc_results['instruction_count'].to_list()
cycles = insn_cyc_results['cycle_count'].to_list()
ipc = insn_cyc_results['IPC'].to_list()

# Program execution metrics
fig1 = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=('Instructions per Second', 'Cycles per Second', 'Instructions per Cycle (IPC)')
)

fig1.add_trace(go.Bar(x=seconds, y=instructions, name='Instructions', marker_color='blue'), row=1, col=1)
fig1.add_trace(go.Bar(x=seconds, y=cycles, name='Cycles', marker_color='green'), row=2, col=1)
fig1.add_trace(go.Scatter(x=seconds, y=ipc, mode='lines+markers', name='IPC', marker_color='red'), row=3, col=1)

fig1.update_layout(
    height=900, showlegend=True, legend=dict(orientation="h", y=1.02),
    xaxis3_title='Time (seconds from start)', 
    yaxis_title='Instruction Count', yaxis2_title='Cycle Count', yaxis3_title='IPC',
    xaxis3=dict(rangeslider=dict(visible=True), type='linear')
)
fig1.show()

# Branch types visualization
seconds_timeline = timeline_analysis['second'].to_list()

fig2 = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=('Branch Types Over Time', 'Branch-to-Instruction Ratios')
)

# Branch counts
fig2.add_trace(go.Scatter(x=seconds_timeline, y=timeline_analysis['branch_count'].to_list(), 
                          mode='lines', name='Branches', line=dict(color='blue')), row=1, col=1)
fig2.add_trace(go.Scatter(x=seconds_timeline, y=timeline_analysis['call_count'].to_list(), 
                          mode='lines', name='Calls', line=dict(color='green')), row=1, col=1)
fig2.add_trace(go.Scatter(x=seconds_timeline, y=timeline_analysis['return_count'].to_list(), 
                          mode='lines', name='Returns', line=dict(color='red')), row=1, col=1)
fig2.add_trace(go.Scatter(x=seconds_timeline, y=timeline_analysis['conditional_count'].to_list(), 
                          mode='lines', name='Conditionals', line=dict(color='purple')), row=1, col=1)

# Branch ratios
fig2.add_trace(go.Scatter(x=seconds_timeline, y=timeline_analysis['branch_ratio'].to_list(), 
                          mode='lines', name='Branch Ratio', line=dict(color='blue')), row=2, col=1)
fig2.add_trace(go.Scatter(x=seconds_timeline, y=timeline_analysis['call_ratio'].to_list(), 
                          mode='lines', name='Call Ratio', line=dict(color='green')), row=2, col=1)
fig2.add_trace(go.Scatter(x=seconds_timeline, y=timeline_analysis['return_ratio'].to_list(), 
                          mode='lines', name='Return Ratio', line=dict(color='red')), row=2, col=1)
fig2.add_trace(go.Scatter(x=seconds_timeline, y=timeline_analysis['conditional_ratio'].to_list(), 
                          mode='lines', name='Conditional Ratio', line=dict(color='purple')), row=2, col=1)

fig2.update_layout(
    height=800, showlegend=True, legend=dict(orientation="h", y=1.02),
    xaxis2_title='Time (seconds from start)', yaxis_title='Count', 
    yaxis2_title='Ratio (branches/instructions)',
    xaxis2=dict(rangeslider=dict(visible=True), type='linear')
)
fig2.show()